# Problem Set: An automated first-pass ontology design review
### Chapter 5 — Methods and Methodologies

**Topic coverage:** Keet §5.1 (methodologies: METHONTOLOGY, On-To-Knowledge, DILIGENT, NeOn, SAMOD; competency questions), §5.2 (ontology quality, OntoClean meta-properties and taxonomy constraints); course notebooks 01–03 of this chapter, and the Chapter 3 tableau. Tools: DSPy + GEPA, LangChain agents, the Chapter 5 methodology recommender and OntoClean checker, value iteration.

**Level:** Graduate (practice-first). Combines conceptual analysis with building, measuring and optimising a real Claude-backed agent.

**Total points:** 100 (Part A 20 / Part B 34 / Part C 27 / Part D 19); 55 pts are auto-graded by the check cells, the rest are marked by hand against the rubric. **Estimated effort:** 10–12 hours.

**API budget:** ≈ \$6–13 estimated on `claude-opus-5` for a full run (≈ 170 DSPy calls and 16 agent episodes; GEPA and the agent runs are the largest lines). Nothing here has been measured — your cost lines are the measurement. Re-running unchanged cells is free: DSPy caches identical requests. Every model call in this notebook is a billed request to Anthropic; the cost lines the problems ask for are part of the answer.

## The brief

**Lattice & Loom** is an ontology consultancy. Every quarter its **design review
board** reads client submissions before any modelling work is quoted: a one-paragraph
project brief, and the draft top-level taxonomy from the client's modelling workshop
together with the workshop's **meta-property tag sheet** (rigidity, identity, unity,
dependence for every class). For each submission the board answers two questions:

* *Which methodology should this project follow?* (§5.1) — decided from the signals
  in the brief, not from habit;
* *Which subsumptions in the draft are ontologically wrong?* (§5.2) — OntoClean
  breaches such as a role placed above a kind (`Organisation <= Customer`) or an
  object filed under the stuff it is made of (`Statue <= Clay`).

The second question is the reason the board exists. Every draft that reaches it has
already passed the client's DL reasoner: the taxonomies are **consistent**, and a
reasoner will never flag them. Dr Tomasz Lindqvist, head of quality, wants a Claude-based
**first-pass auditor** that drafts the board's verdict for each submission. Your brief:

1. **Show the gap.** Demonstrate on the review corpus that a logical reasoner sees none
   of these errors — and that it *propagates* them — and find where the keyword-based
   methodology recommender used to label the corpus stops being trustworthy.
2. **Build the grader first**: a parser for the auditor's output and a diagnostic
   scorer whose feedback names the breached constraint.
3. **Build and optimise the auditor** with DSPy and GEPA — train/dev/test by
   submission, a fixed budget, the cost of every run, and the run-to-run noise.
4. **Ship an audit agent** and measure what its tools buy it, and **plan the client
   projects** as an MDP with prerequisites: when are evaluation and reuse worth their
   cost?

The corpus (24 submissions from 24 clients, split 8/8/8 by submission, every
split containing every methodology and every OntoClean phenomenon), the tag sheet,
the reasoner wiring, the agent's tools and the planning MDP are provided in
`ch05_agentic.py`. Every gold label is re-derived by the chapter's own engines when
the module is imported.

## Submission requirements

1. **This notebook, executed top to bottom**, with every `TODO` implemented and every *Your answer* cell written.
2. **The artefacts it saves** under `course/artifacts/` (optimised instructions, run logs) — they are evidence, not by-products.
3. **Honest reporting.** A number without its split, its sample size and its cost earns no credit. Report the runs you did, including the ones that did not improve anything.

The check cells are the marking scheme for the auto-graded points; do not edit them. The final self-test prints your auto-graded subtotal.

> Solutions: `04_solutions.ipynb` — do not open it until you have attempted every problem.

In [ ]:
import sys, json, logging
from pathlib import Path

# Make the course root and this chapter importable no matter where Jupyter started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break
sys.path.insert(0, str(here))

import oe_course
from oe_course import config
from oe_course.grading import Grader

print(json.dumps(oe_course.describe_environment(), indent=1))
config.require_credentials()          # this problem set calls the live Anthropic API
logging.getLogger("dspy").setLevel(logging.WARNING)
GRADER = Grader('Chapter 5 — Methods and Methodologies — An automated first-pass ontology design review')

import re, time
import dspy
import numpy as np
import pandas as pd
import ch05_toolkit as ch5
import ch05_agentic as A
from oe_course import evaluation as ev, llm, mdp, optimize as opt, agents

train, dev, test = A.build_dataset("train"), A.build_dataset("dev"), A.build_dataset("test")
ALL = train + dev + test
print(f"train {len(train)} / dev {len(dev)} / test {len(test)}")
pd.DataFrame([{"split": ex.split, "item": ex.id, "client": ex.organisation,
               "methodology": ex.gold_methodology, "violations": len(ex.gold_violations)}
              for ex in ALL])

One submission, as the auditor will see it (`brief`, `taxonomy`, `meta_properties`) and
with its gold labels:

In [ ]:
ex = train[0]
print("CLIENT   :", ex.organisation)
print("BRIEF    :", ex.brief, "\n")
print("TAXONOMY :\n" + ex.taxonomy, "\n")
print("TAG SHEET:\n" + ex.meta_properties, "\n")
print("GOLD     :", ex.gold_methodology, ex.gold_violations)
for d in ex.gold_detail:
    print("   ", d["constraint"], "--", d["detail"])

---

## Part A — Consistency is not correctness; signals are not judgement

Two claims underpin the whole review. First, the errors the board looks for are
invisible to a logical reasoner. Second, the gold methodology labels — produced by
`ch5.recommend_methodology` and then reviewed — are only as good as the recommender's
signals. Test both before you let a model near the corpus. Part A calls no model, so it
may look at all 24 submissions.

### A1. What the reasoner sees — and what it spreads (10 pts)

For every submission, load its taxonomy into the Chapter 3 tableau (`A.dl_tbox`,
`A.reasoner_report`, `A.dl.classify`) and build a DataFrame `a1` with one row per
submission and columns:

* `item`, `split`;
* `consistent` — every class in the taxonomy is satisfiable;
* `violating_axioms` — how many axioms `A.taxonomy_violations` flags;
* `entailed` — how many subsumptions between the taxonomy's named classes the
  reasoner entails (`A.dl.classify` over the taxonomy's classes);
* `propagated` — how many of those entailed subsumptions hold **only because of** the
  violating axioms (they disappear when you remove them) **and are not themselves
  asserted**.

Then answer in writing: (a) why can no DL reasoner, however complete, flag these axioms
— what kind of statement is "`Customer` is anti-rigid", and why does it not fit into a
TBox of atomic subsumptions? (b) Take one submission with `propagated > 0` and say
concretely what a downstream SPARQL query over that ontology would get wrong.

In [ ]:
def taxonomy_classes(pairs):
    return sorted({c for pair in pairs for c in pair})

# TODO: one row per submission in ALL
a1 = None

raise NotImplementedError

In [ ]:
# Marking scheme for A1 -- do not edit.
with GRADER.check('A1', points=6):
    assert isinstance(a1, pd.DataFrame) and len(a1) == len(ALL) == 24
    assert {"item", "split", "consistent", "violating_axioms", "entailed", "propagated"} <= set(a1.columns)
    assert a1["consistent"].all(), "every submission is logically consistent"
    _r = a1.set_index("item")
    for _ex in ALL:
        assert _r.loc[_ex.id, "violating_axioms"] == len(_ex.gold_violations), _ex.id
        assert _r.loc[_ex.id, "entailed"] >= len(A.parse_taxonomy(_ex.taxonomy)), _ex.id
    assert _r.loc["port-cargo", "propagated"] == 1          # Container <= PhysicalObject
    assert _r.loc["civic-archives", "propagated"] == 2      # Castle <= Building, Castle <= PhysicalObject
    assert _r.loc["law-firm-pilot", "propagated"] == 2
    assert (_r.loc[_r.violating_axioms == 0, "propagated"] == 0).all()

**Your answer:**

*(Write your answer here.)*

### A2. Where the methodology recommender stops being trustworthy (10 pts)

The corpus's gold methodology is whatever `ch5.recommend_methodology` returns — it counts
signal phrases in the brief — **after** human review: every brief was written and
checked so that the recommender's answer is also the expert's answer.

(a) Write at least three short, realistic briefs on which the recommender is **wrong** by
expert judgement. Store them in `adversarial`, a list of dicts with keys `brief`,
`expert` (a methodology id) and `kind` (your one-word name for the failure — e.g.
`"negation"`); use at least two different kinds.

(b) Build `a2`: one row per corpus submission with columns `item`, `split`, `gold`,
`runner_up` (second-ranked id) and `margin` (top score minus runner-up score).

(c) In writing: given (a), how can the review board justify using the recommender to
label the corpus, and what does (b) tell you about how fragile each label is? When a
live Claude run disagrees with a gold label, what is your procedure — and why is
"optimise the auditor until it agrees" the wrong one?

In [ ]:
adversarial = [
    # {"brief": "...", "expert": "samod", "kind": "..."},
]
a2 = None     # TODO (b)

raise NotImplementedError

In [ ]:
# Marking scheme for A2 -- do not edit.
with GRADER.check('A2', points=5):
    assert isinstance(adversarial, list) and len(adversarial) >= 3
    assert len({a["kind"] for a in adversarial}) >= 2, "use at least two kinds of failure"
    for _a in adversarial:
        assert _a["expert"] in A.METHODOLOGY_IDS, _a
        assert ch5.recommend_methodology(_a["brief"])["recommended"] != _a["expert"], \
            f"the recommender gets this one right: {_a['brief'][:60]}"
    assert isinstance(a2, pd.DataFrame) and len(a2) == 24
    assert {"item", "split", "gold", "runner_up", "margin"} <= set(a2.columns)
    assert (a2["margin"] >= 1).all() and (a2["gold"] != a2["runner_up"]).all()
    _m = a2.set_index("item")["margin"]
    assert _m["citizen-app"] == 2 and _m["port-cargo"] == 4 and _m["carrow-estate"] == 2

**Your answer:**

*(Write your answer here.)*

---

## Part B — Build the grader and the auditor

The guidelines a scorer can report are in `A.AUDIT_RULEBOOK` — ids plus the sentence
GEPA's reflection step will read:

In [ ]:
for rule in A.AUDIT_RULEBOOK:
    print(f"{rule.id:30s} {rule.description[:90]}")

### B1. Parse what the auditor says (8 pts)

Models format answers in many ways; a grader must be strict about *meaning* and lenient
about *notation*. Implement:

* `normalise_axiom(text) -> str | None` — accept `Sub <= Super`, `Sub ⊑ Super`,
  `Sub subClassOf Super`, `Sub rdfs:subClassOf Super` and `Sub is-a Super` (keywords
  case-insensitive, whitespace and surrounding quotes/backticks ignored; class names are
  identifiers `[A-Za-z][A-Za-z0-9_]*`) and return the canonical `"Sub <= Super"`; else
  `None`.
* `parse_violations(value) -> list[str] | None` — `value` is a list, or a string holding a
  JSON array (possibly inside a code fence). Items are axiom strings or objects with an
  `"axiom"` key. `""`, `"none"` and `"[]"` mean no violations (`[]`). Return the
  canonical axioms, de-duplicated in order; return `None` (unparseable) for `None`, for a
  string that is not a JSON array, or if **any** item is not an axiom.
* `methodology_id(text) -> str | None` — map a methodology as written (`"NeOn"`,
  `"On-To-Knowledge"`, `"The NeOn methodology"`) to its catalogue id: compare
  case-insensitively with non-alphanumerics removed against every id and name in
  `ch5.METHODOLOGIES`; failing an exact match, accept the answer if **exactly one**
  catalogue key occurs in it; else `None`.

In [ ]:
def normalise_axiom(text) -> str | None:
    # TODO
    raise NotImplementedError

def parse_violations(value) -> list[str] | None:
    # TODO
    raise NotImplementedError

def methodology_id(text) -> str | None:
    # TODO
    raise NotImplementedError

In [ ]:
# Marking scheme for B1 -- do not edit.
with GRADER.check('B1', points=5):
    assert normalise_axiom("Person <= Student") == "Person <= Student"
    assert normalise_axiom("  `Person ⊑ Student` ") == "Person <= Student"
    assert normalise_axiom("Statue rdfs:subClassOf Clay") == "Statue <= Clay"
    assert normalise_axiom("Statue SUBCLASSOF Clay") == "Statue <= Clay"
    assert normalise_axiom("Pet is-a Animal") == "Pet <= Animal"
    assert normalise_axiom("Person is a kind of Student") is None
    assert parse_violations('["Person <= Student"]') == ["Person <= Student"]
    assert parse_violations('```json\n["Person ⊑ Student", "Statue subClassOf Clay"]\n```') == \
        ["Person <= Student", "Statue <= Clay"]
    assert parse_violations('[{"axiom": "Person rdfs:subClassOf Pet", "constraint": "x"}]') == ["Person <= Pet"]
    assert parse_violations(["Person <= Student", "Person<=Student"]) == ["Person <= Student"]
    assert parse_violations("[]") == [] and parse_violations("none") == [] and parse_violations("") == []
    assert parse_violations("Person <= Student") is None
    assert parse_violations('["Person <= Student", "looks fine"]') is None
    assert parse_violations(None) is None and parse_violations('{"axiom": "A <= B"}') is None
    assert methodology_id("NeOn") == "neon" and methodology_id("  METHONTOLOGY ") == "methontology"
    assert methodology_id("On-To-Knowledge") == "on-to-knowledge" == methodology_id("on to knowledge")
    assert methodology_id("The NeOn methodology") == "neon" and methodology_id("SAMOD (agile)") == "samod"
    assert methodology_id("NeOn or DILIGENT") is None and methodology_id("waterfall") is None
    assert methodology_id("") is None and methodology_id(None) is None

### B2. A diagnostic audit scorer (12 pts)

Implement `audit_scorer(gold, pred) -> ev.ScoreReport`, where `gold` is a dataset row and
`pred` has `methodology` and `violations`. Score `0.4 × methodology_ok + 0.6 ×
violation_F1` (the taxonomy carries more weight: it is the part a reasoner cannot do).

| situation | effect | `violated` |
|---|---|---|
| `methodology_id(pred.methodology)` is `None` | methodology 0 | `emit-structured-audit` **and** `choose-<gold>` |
| a catalogue id, but not gold | methodology 0 | `choose-<gold>` |
| `parse_violations(...)` is `None` | F1 = 0; no further taxonomy diagnosis | `emit-structured-audit` |
| a gold violation not reported | lowers F1 | `apply-ontoclean-constraints` |
| a **sound** axiom of the taxonomy reported | lowers F1 | `only-report-real-violations` |
| an axiom that is **not in the taxonomy** reported (e.g. reversed) | lowers F1 | `only-audit-given-axioms` |

F1 is `ev.set_f1(reported, gold.gold_violations)` (a clean taxonomy answered `[]` scores
F1 = 1). `violated` lists each id at most once. The `notes` are what GEPA reads, so make
them specific: for a missed axiom, the checker's `detail` and `constraint` from
`gold.gold_detail`; for a sound axiom reported, both classes' tags from `A.TAG_SHEET`
and why no constraint applies; for a wrong methodology, the answer, the gold id and the
brief's decisive signals.

In [ ]:
W_METHOD, W_VIOLATIONS = 0.4, 0.6

def tags(name: str) -> str:
    t = A.TAG_SHEET[name]
    return f"{t.rigidity} {t.identity} {t.unity} {t.dependence}"

def audit_scorer(gold, pred) -> ev.ScoreReport:
    # TODO
    raise NotImplementedError

In [ ]:
# Marking scheme for B2 -- do not edit.
with GRADER.check('B2', points=9):
    gold = next(ex for ex in dev if ex.id == "carrow-estate")      # methontology; Door <= Doorway
    clean = next(ex for ex in dev if ex.id == "warehouse-robotics")  # samod; no violations
    P = lambda m, v: dspy.Prediction(methodology=m, violations=v)
    cases = [
        (gold, P("METHONTOLOGY", '["Door <= Doorway"]'), 1.0, set()),
        (gold, P("methontology", '["Door ⊑ Doorway"]'), 1.0, set()),
        (gold, P("NeOn", '["Door <= Doorway"]'), 0.6, {"choose-methontology"}),
        (gold, P("waterfall", '["Door <= Doorway"]'), 0.6, {"emit-structured-audit", "choose-methontology"}),
        (gold, P("methontology", "[]"), 0.4, {"apply-ontoclean-constraints"}),
        (gold, P("methontology", '["Door <= Doorway", "Tenant <= Person"]'), 0.4 + 0.6 * 2 / 3,
         {"only-report-real-violations"}),
        (gold, P("methontology", '["Doorway <= Door"]'), 0.4,
         {"only-audit-given-axioms", "apply-ontoclean-constraints"}),
        (gold, P("methontology", "Door <= Doorway"), 0.4, {"emit-structured-audit"}),
        (gold, P("fake", None), 0.0, {"emit-structured-audit", "choose-methontology"}),
        (clean, P("SAMOD", "[]"), 1.0, set()),
        (clean, P("samod", '["Operator <= Person"]'), 0.4, {"only-report-real-violations"}),
    ]
    for g, p, score, violated in cases:
        r = audit_scorer(g, p)
        assert abs(r.score - score) < 1e-6, (p, r.score, score)
        assert set(r.violated) == violated and len(r.violated) == len(set(r.violated)), (p, r.violated)
        assert all(v in A.AUDIT_RULEBOOK for v in r.violated)
    missed = audit_scorer(gold, P("methontology", "[]"))
    assert any("dependent-cannot-subsume-independent" in n for n in missed.notes), "name the constraint"
    sound = audit_scorer(gold, P("methontology", '["Door <= Doorway", "Tenant <= Person"]'))
    assert any("Tenant" in n and "+D" in n for n in sound.notes), "explain why the axiom is sound"

### B3. The auditor as a DSPy program (6 pts)

Write a signature `AuditSubmission` with inputs `brief`, `taxonomy`, `meta_properties`
and outputs `methodology`, `violations` and a one-sentence `justification`, and a
factory `AuditProgram(instruction)` returning a `dspy.Module` with a single
`dspy.Predict` whose instruction is `instruction`. The field descriptions are part of
the prompt (manual marks): say what format `violations` must take. Configure Claude and
run the program once on `train[0]`.

In [ ]:
BASELINE_INSTRUCTION = A.BASELINE_INSTRUCTION

# TODO: class AuditSubmission(dspy.Signature): ...
# TODO: def AuditProgram(instruction=BASELINE_INSTRUCTION): ...

lm = llm.configure_dspy()
smoke = None     # TODO: AuditProgram()(**train[0].inputs())

raise NotImplementedError

In [ ]:
# Marking scheme for B3 -- do not edit.
with GRADER.check('B3', points=3):
    assert set(AuditSubmission.input_fields) == {"brief", "taxonomy", "meta_properties"}
    assert {"methodology", "violations"} <= set(AuditSubmission.output_fields)
    _program = AuditProgram("custom instruction")
    assert len(list(_program.named_predictors())) == 1
    assert opt.instruction_of(_program) == "custom instruction"
    assert isinstance(smoke.methodology, str)

### B4. Baseline on the development split, with its cost (8 pts)

Evaluate the baseline program on `dev` with `ev.evaluate_dataset`, inside `llm.meter(lm)`.
Store the result in `baseline_dev` and the cost in `baseline_dev_cost`; show the per-item
rows.

Then write an **error analysis**: for each non-perfect item, which guideline was violated
and why you think Claude did it. Keep three kinds of failure apart — format (the
contract), methodology (reading the brief), and OntoClean (reading the tag sheet) — and
say which one the scorer's weights make most expensive.

In [ ]:
# TODO: baseline_dev = ..., baseline_dev_cost = ...

raise NotImplementedError

In [ ]:
# Marking scheme for B4 -- do not edit.
with GRADER.check('B4', points=3):
    assert baseline_dev["n"] == len(dev) == 8
    assert {r["item"] for r in baseline_dev["rows"]} == {ex.id for ex in dev}
    assert 0.0 <= baseline_dev["mean_score"] <= 1.0
    assert {"calls", "usd"} <= set(baseline_dev_cost)

**Your answer:**

*(Write your answer here.)*

---

## Part C — Optimise with GEPA — and report it honestly

GEPA rewrites the instruction by reflecting on the scorer's feedback. It can only learn
a guideline that the training data gives the program a chance to break, it overfits
small training sets, and every metric call is billed. The rules: **optimise on `train`,
select on `dev`, report on `test`** — with a cost and a noise estimate next to every
number.

### C1. Which guidelines can the training data teach? (7 pts)

A guideline only appears in GEPA's feedback if some training answer *can* violate it.
Implement `can_violate(rule_id, ex) -> bool`: is there **some** answer to `ex` for which
your `audit_scorer` would report `rule_id`? Then:

1. Build `c1`: a DataFrame indexed by rule id (every rule in `A.AUDIT_RULEBOOK`) with
   columns `train`, `dev`, `test` — the number of items in each split on which the rule
   can be violated.
2. Build `ablated_train`: `train` with every **sound** axiom removed from each taxonomy
   (and `meta_properties` restricted to the classes that remain), dropping items left
   with no axioms. Add its counts as a column `ablated_train`.

In writing: predict what GEPA would learn from `ablated_train` and what would happen on
`test`. Why is this the dangerous kind of data change? State the general principle.

In [ ]:
def can_violate(rule_id: str, ex) -> bool:
    # TODO
    raise NotImplementedError

c1 = None              # TODO (1)
ablated_train = None   # TODO (2), then add the column to c1

In [ ]:
# Marking scheme for C1 -- do not edit.
with GRADER.check('C1', points=5):
    assert isinstance(c1, pd.DataFrame) and set(c1.index) == set(A.AUDIT_RULEBOOK.ids)
    assert {"train", "dev", "test", "ablated_train"} <= set(c1.columns)
    for _split, _rows in [("train", train), ("dev", dev), ("test", test)]:
        assert c1.loc["apply-ontoclean-constraints", _split] == sum(bool(e.gold_violations) for e in _rows)
        for _m in A.METHODOLOGY_IDS:
            assert c1.loc[f"choose-{_m}", _split] == sum(e.gold_methodology == _m for e in _rows)
        assert c1.loc["emit-structured-audit", _split] == len(_rows)
    assert c1.loc["only-report-real-violations", "train"] == 8
    assert c1.loc["only-report-real-violations", "ablated_train"] == 0
    assert len(ablated_train) == 7 and all(e.gold_violations for e in ablated_train)
    assert all(set(e.inputs().keys()) == {"brief", "taxonomy", "meta_properties"} for e in ablated_train)
    for _e in ablated_train:
        _pairs = A.parse_taxonomy(_e.taxonomy)
        assert {A.format_axiom(*p) for p in _pairs} == set(_e.gold_violations)
        assert set(_e.meta_properties.split("\n")) == set(A.meta_properties_text(_pairs).split("\n"))
        # no answer on an ablated item can be marked as over-reporting
        _all = dspy.Prediction(methodology=_e.gold_methodology, violations=[A.format_axiom(*p) for p in _pairs])
        assert "only-report-real-violations" not in audit_scorer(_e, _all).violated

**Your answer:**

*(Write your answer here.)*

### C2. A budgeted GEPA run, reported on test (12 pts)

1. Build the GEPA feedback metric from your scorer and `A.AUDIT_RULEBOOK`, and a separate
   reflection LM (`llm.reflection_lm()`).
2. Run `opt.run_gepa` on `train` with `valset=dev` and `max_metric_calls=GEPA_BUDGET`
   inside `llm.meter(lm, reflect)`; store the cost in `gepa_cost`.
3. Save the tuned instruction to `config.artifacts_dir() / "ch05_audit_instruction.txt"`.
4. Add the **hand-written guidelines** contender,
   `A.AUDIT_RULEBOOK.as_guidelines(BASELINE_INSTRUCTION)`, and evaluate baseline,
   guidelines and GEPA-tuned on **`test`**, each inside its own `llm.meter`. Build `c2`
   with columns `program` (`"baseline"`, `"guidelines"`, `"gepa"`), `test_mean`,
   `violations`, `eval_usd`, `optimisation_usd` (GEPA's cost for `"gepa"`, 0 otherwise)
   and `instruction_chars`.

In writing: read the instruction diff. Which guidelines did GEPA put into words, which did
it miss (compare with `c1`), and did it add anything that only fits the eight training
clients? Which program would you deploy, and what would change your mind?

In [ ]:
GEPA_BUDGET = 60
instruction_path = config.artifacts_dir() / "ch05_audit_instruction.txt"
guidelines_instruction = A.AUDIT_RULEBOOK.as_guidelines(BASELINE_INSTRUCTION)
# TODO: gepa_metric, reflect, tuned (inside llm.meter -> gepa_cost), save, c2
c2 = None

raise NotImplementedError

In [ ]:
# Marking scheme for C2 -- do not edit.
with GRADER.check('C2', points=5):
    _ids = lambda rows: {ex.id for ex in rows}
    assert not (_ids(test) & (_ids(train) | _ids(dev))), "the test split must be untouched"
    assert instruction_path.is_file()
    assert instruction_path.read_text(encoding="utf-8") == opt.instruction_of(tuned)
    assert {"calls", "usd"} <= set(gepa_cost) and GEPA_BUDGET <= 100
    assert isinstance(c2, pd.DataFrame) and set(c2["program"]) == {"baseline", "guidelines", "gepa"}
    assert {"test_mean", "violations", "eval_usd", "optimisation_usd", "instruction_chars"} <= set(c2.columns)
    assert c2.set_index("program").loc[["baseline", "guidelines"], "optimisation_usd"].eq(0).all()
    assert c2["test_mean"].between(0, 1).all()

**Your answer:**

*(Write your answer here.)*

### C3. Is the difference bigger than the noise? (8 pts)

Eight test items at temperature 1.0: one item changes a mean by up to 0.125. With caching
**disabled** (`fresh = llm.dspy_lm(cache=False)` and `with dspy.context(lm=fresh): ...`),
run the baseline and the GEPA-tuned programs on `test` three times each, inside
`llm.meter(fresh)` (store as `c3_cost`); store the mean scores in
`runs = {"baseline": [..3..], "gepa": [..3..]}`.

Set `verdict_c3` to `"significant"` if `|mean_gepa − mean_baseline| > 2 ·
max(sd_gepa, sd_baseline)` (sample standard deviations), else `"not significant"`. In
writing: what does this say about the C2 table, and what would a credible evaluation of
the auditor need?

In [ ]:
fresh = llm.dspy_lm(cache=False)
runs = {"baseline": [], "gepa": []}
# TODO: fill runs inside llm.meter(fresh) as c3_cost, then compute verdict_c3
verdict_c3 = None

raise NotImplementedError

In [ ]:
# Marking scheme for C3 -- do not edit.
with GRADER.check('C3', points=4):
    import statistics as _st
    assert set(runs) == {"baseline", "gepa"} and all(len(v) == 3 for v in runs.values())
    assert all(0.0 <= x <= 1.0 for v in runs.values() for x in v)
    assert {"calls", "usd"} <= set(c3_cost) and c3_cost["calls"] >= 2 * 3 * len(test)
    _mean = {k: _st.mean(v) for k, v in runs.items()}
    _sd = {k: _st.stdev(v) for k, v in runs.items()}
    assert verdict_c3 == ("significant" if abs(_mean["gepa"] - _mean["baseline"]) > 2 * max(_sd.values())
                          else "not significant")

**Your answer:**

*(Write your answer here.)*

---

## Part D — An audit agent, and planning the project

The agent audits one submission at a time with tools (`A.build_audit_tools`): the
methodology catalogue, the tag-sheet lookup, the OntoClean constraints, a **logical
consistency check** (the Chapter 3 tableau) — and, optionally, the two *engines* that
decide the gold labels: the methodology recommender and the OntoClean taxonomy checker.
The agent receives the brief and the taxonomy (`ws.task()`), not the tag sheet: it must
look tags up.

In [ ]:
_ws = A.AuditWorkspace(A.AUDIT_CASES[0])
for t in A.build_audit_tools(_ws, include_engines=True):
    print(f"{t.name:22s} {t.description.splitlines()[0][:80]}")
print()
print(_ws.task())

### D1. What do the tools buy the agent? (10 pts)

1. Write `AUDITOR_PROMPT`: the agent must look up the tags of **every** class, check
   **every** axiom against the constraints, not treat logical consistency as evidence of
   correctness, and end with a JSON object
   `{"methodology": "<id>", "violations": ["Sub <= Super", ...], "justification": "..."}`.
2. Write `parse_audit(text) -> dspy.Prediction` returning `methodology` and `violations`
   from the **last** JSON object in the text that has a `"methodology"` key (it may be in
   a code fence, surrounded by prose, and contain nested objects), or
   `dspy.Prediction(methodology="", violations=None)` when there is none.
3. For each **test** submission and each toolset — `"engines"` (`include_engines=True`)
   and `"lookup-only"` (`include_engines=False`) — build a fresh `A.AuditWorkspace(case)`,
   its tools, an agent (`agents.build_agent(ws, system_prompt=..., tools=...)`), run it on
   `ws.task()`, and score `parse_audit(run.answer)` with your `audit_scorer`. Collect `d1`
   with columns `id`, `toolset`, `answer`, `score`, `violated`, `tool_calls`,
   `used_reasoner` (did it call `check_consistency`?), `used_engines` (did it call
   `recommend_methodology` or `check_taxonomy`?) and `usd_estimate`
   (`llm.chat_usage(run.messages)`).

In writing: compare the toolsets (mean score with n, cost per submission) and examine
every error. Did the agent call the reasoner, and did "consistent" ever mislead it? With
the engines available the task reduces to calling two tools — what, then, does the agent
add, and which toolset would you let draft verdicts for the board?

In [ ]:
AUDITOR_PROMPT = """TODO"""

def parse_audit(text: str) -> dspy.Prediction:
    # TODO
    raise NotImplementedError

d1 = None   # TODO: 2 toolsets x the test split

In [ ]:
# Marking scheme for D1 -- do not edit.
with GRADER.check('D1', points=4):
    _p = parse_audit('Checked.\n```json\n{"methodology": "NeOn", "violations": ["Pump <= Borehole"]}\n```')
    assert methodology_id(_p.methodology) == "neon" and parse_violations(_p.violations) == ["Pump <= Borehole"]
    _p = parse_audit('{"note": 1} then {"methodology": "samod", "violations": [{"axiom": "Person <= Child"}], "justification": "x"}')
    assert _p.methodology == "samod" and parse_violations(_p.violations) == ["Person <= Child"]
    _p = parse_audit('{"methodology": "diligent", "violations": []} {"methodology": "neon", "violations": []}')
    assert _p.methodology == "neon"
    for _bad in ["I recommend SAMOD.", "{}", "", '{"verdict": "yes"}']:
        _p = parse_audit(_bad)
        assert _p.methodology == "" and _p.violations is None, _bad
    assert isinstance(d1, pd.DataFrame) and len(d1) == 2 * len(test)
    assert set(d1["toolset"]) == {"engines", "lookup-only"}
    assert {"id", "toolset", "answer", "score", "violated", "tool_calls", "used_reasoner",
            "used_engines", "usd_estimate"} <= set(d1.columns)
    assert not d1.loc[d1.toolset == "lookup-only", "used_engines"].any(), "lookup-only has no engines"
    _gold = {ex.id: ex for ex in test}
    for _, _row in d1.iterrows():
        assert abs(audit_scorer(_gold[_row.id], parse_audit(_row.answer)).score - _row.score) < 1e-9

**Your answer:**

*(Write your answer here.)*

### D2. Planning the project: when are evaluation and reuse worth it? (9 pts)

`A.MethodologyPlanMDP` plans a development project over `ch5.DEVELOPMENT_STEPS`: a step
is available only once its prerequisites are complete, each step costs its effort, and
`ship` pays the **measured** competency-question coverage of the partial AWO
(`A.measured_coverage`). Hooks: `coverage_fn(done)` and `cost_fn(done, action)`.

1. Solve it with `mdp.value_iteration`, roll out the greedy policy, and store `plan` (the
   list of actions), `v_star` (the value of the initial state) and `reachable` — the
   number of *unshipped* states reachable from the start (of the 128 subsets of steps).
2. **Evaluation.** Suppose shipping without `evaluation` delivers only `(1 − q)` of the
   coverage (defects found later). Derive, on paper, the `q*` above which the optimal
   plan evaluates, implement `evaluation_threshold(eval_cost, coverage)`, and verify it:
   sweep `q` over `np.linspace(0, 0.3, 31)` and record `d2_eval` (columns `q`,
   `evaluates`).
3. **Reuse (NeOn).** Suppose that once `reuse_search` is done, `taxonomy` and `axioms`
   cost `(1 − d)` of their listed cost. Derive `d*`, implement
   `reuse_threshold(search_cost, discountable_cost)`, and verify it with a sweep over
   `np.linspace(0, 0.8, 17)` recorded in `d2_reuse` (columns `discount`, `searches`).

In writing: what does `reachable` say about what a methodology is? What do the two
thresholds say about the steps real teams skip, and whose fault is it when an agent (or a
team) optimising a measured objective skips evaluation?

In [ ]:
M = A.MethodologyPlanMDP()
plan, v_star, reachable = None, None, None        # TODO (1)

def evaluation_threshold(eval_cost: float, coverage: float) -> float:
    # TODO (2)
    raise NotImplementedError

def reuse_threshold(search_cost: float, discountable_cost: float) -> float:
    # TODO (3)
    raise NotImplementedError

d2_eval, d2_reuse = None, None                    # TODO (2), (3)

In [ ]:
# Marking scheme for D2 -- do not edit.
with GRADER.check('D2', points=6):
    assert abs(v_star - 0.25) < 1e-9, "coverage 1.0 minus 0.75 of effort"
    assert plan[-1] == "ship"
    assert set(plan[:-1]) == {"requirements", "competency_questions", "taxonomy", "axioms", "instances"}
    _done = set()
    for _a in plan[:-1]:
        assert all(r in _done for r in ch5.DEVELOPMENT_STEPS[_a]["requires"]), f"{_a} before its prerequisites"
        _done.add(_a)
    assert reachable == 21
    assert abs(evaluation_threshold(0.10, 1.0) - 0.10) < 1e-9 and abs(evaluation_threshold(0.10, 0.5) - 0.20) < 1e-9
    assert abs(reuse_threshold(0.15, 0.45) - 1 / 3) < 1e-9
    _q = evaluation_threshold(0.10, 1.0)
    _clear = d2_eval[(d2_eval.q - _q).abs() > 0.015]
    assert len(_clear) >= 25 and (_clear.evaluates == (_clear.q > _q)).all(), "sweep disagrees with q*"
    _d = reuse_threshold(0.15, 0.45)
    _clear = d2_reuse[(d2_reuse.discount - _d).abs() > 0.03]
    assert len(_clear) >= 15 and (_clear.searches == (_clear.discount > _d)).all(), "sweep disagrees with d*"

**Your answer:**

*(Write your answer here.)*

---

## Grading rubric

| Part | Problem | Auto | Manual | Pts |
|---|---|---|---|---|
| A | A1 What the reasoner sees — and what it spreads | 6 | 4 | 10 |
| A | A2 Where the methodology recommender stops being trustworthy | 5 | 5 | 10 |
| B | B1 Parse what the auditor says | 5 | 3 | 8 |
| B | B2 A diagnostic audit scorer | 9 | 3 | 12 |
| B | B3 The auditor as a DSPy program | 3 | 3 | 6 |
| B | B4 Baseline on the development split, with its cost | 3 | 5 | 8 |
| C | C1 Which guidelines can the training data teach? | 5 | 2 | 7 |
| C | C2 A budgeted GEPA run, reported on test | 5 | 7 | 12 |
| C | C3 Is the difference bigger than the noise? | 4 | 4 | 8 |
| D | D1 What do the tools buy the agent? | 4 | 6 | 10 |
| D | D2 Planning the project: when are evaluation and reuse worth it? | 6 | 3 | 9 |
| **Total** | | | | **100** |

*Manual* points are for the written answers and for the quality of the engineering evidence (splits, sample sizes, cost, honest negative results).

---

## Self-test

Re-run this cell after finishing. It lists every auto-graded problem and whether its check passed.

In [ ]:
AUTO_GRADED = ['A1', 'A2', 'B1', 'B2', 'B3', 'B4', 'C1', 'C2', 'C3', 'D1', 'D2']
_ = GRADER.summary(AUTO_GRADED)